In [1]:
from intermine.webservice import Service

In [2]:
import numpy as np
import pandas as pd
import scanpy as sc

In [3]:
import glob
import os

In [4]:
run_dirs = glob.glob('../TX4[56]/*LamC') + glob.glob(f'../TX4[56]/*Prosalpha3') + glob.glob('../TX4[56]/*ctrl')
run_dirs.sort()
run_names = [os.path.basename(rd) for rd in run_dirs]
run_labels = ['TX45 ctrl', 'TX45 LamC', 'TX45 Prosalpha3', 'TX46 ctrl', 'TX46 LamC', 'TX46 Prosalpha3']
list(zip(run_dirs, run_names, run_labels))

[('../TX45/DS-0023A_ctrl', 'DS-0023A_ctrl', 'TX45 ctrl'),
 ('../TX45/DS-0023C_LamC', 'DS-0023C_LamC', 'TX45 LamC'),
 ('../TX45/DS-0023D_Prosalpha3', 'DS-0023D_Prosalpha3', 'TX45 Prosalpha3'),
 ('../TX46/DS-0024E_ctrl', 'DS-0024E_ctrl', 'TX46 ctrl'),
 ('../TX46/DS-00250_LamC', 'DS-00250_LamC', 'TX46 LamC'),
 ('../TX46/DS-00251_Prosalpha3', 'DS-00251_Prosalpha3', 'TX46 Prosalpha3')]

In [5]:
sub_adatas = [
    sc.read_10x_mtx(
    '{}/counts/outs/raw_feature_bc_matrix/'.format(run_dir),  # the directory with the `.mtx` file
    var_names='gene_ids',                # use gene symbols for the variable names (variables-axis index)
    cache=True)                              # write a cache file for faster subsequent reading
    for run_dir in run_dirs
] + [
    sc.read_10x_mtx(
    '{}/counts/outs/filtered_feature_bc_matrix/'.format(run_dir),  # the directory with the `.mtx` file
    var_names='gene_ids',                # use gene symbols for the variable names (variables-axis index)
    cache=True)                              # write a cache file for faster subsequent reading
    for run_dir in run_dirs 
]

In [6]:
all_obs_fbgns = set()
obs_fbgn_given_symbol = {}
for adata in sub_adatas:
    all_obs_fbgns.update(list(adata.var.index))
    for fbgn, symbol in zip(adata.var.index, adata.var.gene_symbols):
        if symbol not in obs_fbgn_given_symbol: 
            obs_fbgn_given_symbol[symbol] = fbgn
        else:
            assert fbgn == obs_fbgn_given_symbol[symbol], (symbol, fbgn, obs_fbgn_given_symbol[symbol])

del sub_adatas
len(all_obs_fbgns), len(obs_fbgn_given_symbol), list(all_obs_fbgns)[:5]

(13942,
 13942,
 ['FBgn0034061', 'FBgn0015010', 'FBgn0002775', 'FBgn0000239', 'FBgn0037304'])

In [7]:
from intermine.webservice import Service
from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests

def fetch_flybase_gene_data(fbgn_ids):
    """
    Fetch GO terms for a list of FBgn IDs from FlyBase via FlyMine.

    Parameters:
        fbgn_ids (list of str): FlyBase Gene IDs

    Returns:
        dict of FBgn ID -> {
            symbol (str),
            go_terms (set of tuples): (GO ID, name, namespace)
        }
    """
    service = Service("https://www.flymine.org/flymine/service")

    query = service.new_query("Gene")
    query.add_view(
        "primaryIdentifier",
        "symbol",
        "goAnnotation.ontologyTerm.identifier",
        "goAnnotation.ontologyTerm.name",
        "goAnnotation.ontologyTerm.namespace",
    )
    query.add_constraint("primaryIdentifier", "ONE OF", fbgn_ids, code="A")

    gene_data = {}
    for row in query.rows():
        fbgn = row["primaryIdentifier"]
        symbol = row["symbol"]
        go_id = row["goAnnotation.ontologyTerm.identifier"]
        go_name = row["goAnnotation.ontologyTerm.name"]
        go_ns = row["goAnnotation.ontologyTerm.namespace"]

        if fbgn not in gene_data:
            gene_data[fbgn] = {"symbol": symbol, "go_terms": set()}

        if go_id and go_name and go_ns:
            gene_data[fbgn]["go_terms"].add((go_id, go_name, go_ns))

    return gene_data


def fetch_background_go_counts(background_fbgn_ids):
    """
    Get background GO term counts for all annotated Drosophila genes.

    Returns:
        dict of GO term -> set of FBgn IDs
    """
    service = Service("https://www.flymine.org/flymine/service")
    query = service.new_query("Gene")
    query.add_view("primaryIdentifier", "goAnnotation.ontologyTerm.identifier")
    query.add_constraint("primaryIdentifier", "ONE OF", background_fbgn_ids, code="A")

    go_to_genes = defaultdict(set)

    for row in query.rows():
        fbgn = row["primaryIdentifier"]
        go_id = row["goAnnotation.ontologyTerm.identifier"]
        if fbgn and go_id:
            go_to_genes[go_id].add(fbgn)

    return go_to_genes

def compute_go_enrichment(input_genes_data, go_to_genes, q_thresh=0.05):
    """
    Compute enrichment of GO terms in the input gene list.

    Returns:
        list of enriched terms with statistics
    """
    input_fbgns = set(input_genes_data.keys())
    go_counts_input = defaultdict(int)

    # Count GO terms in input list
    for fbgn, info in input_genes_data.items():
        for go_id, go_name, go_ns in info["go_terms"]:
            go_counts_input[go_id] += 1

    enrichment_results = []
    all_input = len(input_fbgns)
    all_background = len({g for genes in go_to_genes.values() for g in genes})

    for go_id, count_in_input in go_counts_input.items():
        count_in_background = len(go_to_genes.get(go_id, set()))
        overlap = len(input_fbgns & go_to_genes.get(go_id, set()))
        not_in_input = all_input - overlap
        in_bg_only = count_in_background - overlap
        rest = all_background - all_input - in_bg_only

        contingency = [[overlap, not_in_input],
                       [in_bg_only, rest]]

        _, p_value = fisher_exact(contingency, alternative='greater')
        enrichment_results.append((go_id, count_in_input, count_in_background, p_value))

    # Multiple testing correction
    p_vals = [p for _, _, _, p in enrichment_results]
    if p_vals:
        _, q_vals, _, _ = multipletests(p_vals, method='fdr_bh')
    else:
        q_vals = []

    enriched = []
    for (go_id, in_input, in_bg, p), q in zip(enrichment_results, q_vals):
        if q < q_thresh:
            enriched.append({
                "GO_ID": go_id,
                "Input_Count": in_input,
                "Background_Count": in_bg,
                "p_value": p,
                "q_value": q
            })

    return sorted(enriched, key=lambda x: x["q_value"])

In [8]:
def build_go_info_given_go_id():
    """
    Build a dict giving GO info for all GO terms in annotated Drosophila genes.

    Returns:
        dict of GO ID -> GO info
    """
    service = Service("https://www.flymine.org/flymine/service")
    query = service.new_query("Gene")
    query.add_view(
        "primaryIdentifier",
        "symbol",
        "goAnnotation.ontologyTerm.identifier",
        "goAnnotation.ontologyTerm.name",
        "goAnnotation.ontologyTerm.namespace",
    )
    
    go_info = {}
    for row in query.rows():
        fbgn = row["primaryIdentifier"]
        symbol = row["symbol"]
        go_id = row["goAnnotation.ontologyTerm.identifier"]
        go_name = row["goAnnotation.ontologyTerm.name"]
        go_ns = row["goAnnotation.ontologyTerm.namespace"]

        if go_id not in go_info:
            go_info[go_id] = {
                'go_name': go_name,
                'go_ns': go_ns,
            }
        else:
            assert go_info[go_id]['go_name'] == go_name and go_info[go_id]['go_ns'] == go_ns, (go_id, go_info[go_id], go_name, go_ns)
            
    return go_info

In [9]:
go_info_given_id = build_go_info_given_go_id()

In [10]:
def build_fbgn_given_symbol(fbgn_ids):
    """
    Build a dict giving gene symbol given fbgn id 

    Returns:
        dict of symbol -> fbgn_id
    """
    service = Service("https://www.flymine.org/flymine/service")
    query = service.new_query("Gene")
    query.add_view(
        "primaryIdentifier",
        "symbol",
    )
    query.add_constraint("primaryIdentifier", "ONE OF", fbgn_ids, code="A")

    
    fbgn_given_symbol = {}
    all_fbgn_and_symbol = []
    for row in query.rows():
        fbgn = row["primaryIdentifier"]
        symbol = row["symbol"]
        if symbol is None:
            continue
            
        all_fbgn_and_symbol.append((fbgn, symbol))

        if symbol not in fbgn_given_symbol:
            fbgn_given_symbol[symbol] = fbgn
        else:
            assert fbgn_given_symbol[symbol] == fbgn, (fbgn, symbol, fbgn_given_symbol[symbol])
            
    return fbgn_given_symbol, all_fbgn_and_symbol

In [11]:
fbgn_given_symbol, all_fbgn_and_symbol = build_fbgn_given_symbol(all_obs_fbgns)

In [12]:
from collections import Counter

In [13]:
Counter([symbol for fbgn, symbol in all_fbgn_and_symbol]).most_common(10)

[('zyd', 1),
 ('RhoGAP1A', 1),
 ('CG17167', 1),
 ('CG12061', 1),
 ('CG41099', 1),
 ('CG17707', 1),
 ('CG17715', 1),
 ('Alg-2', 1),
 ('Set1', 1),
 ('su(f)', 1)]

In [14]:
len([fbgn for fbgn, symbol in all_fbgn_and_symbol if fbgn.startswith('FB')])

13898

In [15]:
len([fbgn for fbgn, symbol in all_fbgn_and_symbol])

13898

In [16]:
fbgn_ids = ["FBgn0004647", "FBgn0000181", "FBgn0000017"]  # Sxl, ftz, bcd

gene_data = fetch_flybase_gene_data(fbgn_ids)
background_go = fetch_background_go_counts(all_obs_fbgns)

enriched = compute_go_enrichment(gene_data, background_go)

for item in enriched[:10]:  # top 10
    go_info = go_info_given_id[item['GO_ID']]
    print(f"{item['GO_ID']}: input={item['Input_Count']} background={item['Background_Count']:4,d} "
          f"p={item['p_value']:.3e}, q={item['q_value']:.3e}"
          f"   {go_info['go_ns']}, {go_info['go_name']}")


GO:0014018: input=1 background=   3 p=7.237e-04, q=5.066e-03   biological_process, neuroblast fate specification
GO:0007391: input=2 background=  98 p=1.835e-04, q=5.066e-03   biological_process, dorsal closure
GO:0060571: input=1 background=   2 p=4.825e-04, q=5.066e-03   biological_process, morphogenesis of an epithelial fold
GO:0048627: input=1 background=   2 p=4.825e-04, q=5.066e-03   biological_process, myoblast development
GO:0009608: input=1 background=   4 p=9.649e-04, q=5.066e-03   biological_process, response to symbiont
GO:0048052: input=1 background=   3 p=7.237e-04, q=5.066e-03   biological_process, R1/R6 cell differentiation
GO:2000048: input=1 background=   1 p=2.413e-04, q=5.066e-03   biological_process, negative regulation of cell-cell adhesion mediated by cadherin
GO:0042686: input=1 background=   1 p=2.413e-04, q=5.066e-03   biological_process, regulation of cardioblast cell fate specification
GO:0060289: input=1 background=   3 p=7.237e-04, q=5.066e-03   biological

# GO analysis for DE genes

In [17]:
def read_this_csv(fpath):
    entries = []
    for i, line in enumerate(open(fpath)):
        if i == 0:
            headers = line.strip().split(',')
            headers[0] = 'symbol'
        else:
            entries.append({h: v for h, v in zip(headers, line.strip().split(','))})
    return entries

In [18]:
def write_enriched_go_csv(enriched_go, out_fpath):
    with open(out_fpath, 'w') as out:
        out.write('GO ID,pval,qval,Obs count,Background count,GO Cat,GO Name\n')
        for item in enriched_go:
            go_info = go_info_given_id[item['GO_ID']]
            out.write(','.join([item['GO_ID'], 
                                f"{item['p_value']:.3e},{item['q_value']:.3e}",
                                str(item['Input_Count']), str(item['Background_Count']), 
                                go_info['go_ns'], go_info['go_name'], ]) + '\n')

In [19]:
background_go = fetch_background_go_counts(all_obs_fbgns)

In [20]:
#DE_dirs = [d for d in glob.glob('results/*') if 'GO' not in d]
gois = ['Prosalpha3', 'LamC']
DE_dirs = [f'results/{goi}' for goi in gois]
DE_dirs

['results/Prosalpha3', 'results/LamC']

In [21]:
for dpath in DE_dirs:
    out_dir = dpath + '_GO'
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    fpaths = glob.glob(f'{dpath}/*csv')
    for fpath in fpaths:
        print(fpath)
        out_fpath = os.path.join(out_dir, os.path.basename(fpath))
        print(out_fpath)
        symbols = [entry['symbol'] for entry in read_this_csv(fpath)]
        print(symbols)
        fbgn_ids = [fbgn_given_symbol[symbol] for symbol in symbols]
        print(fbgn_ids)
        gene_data = fetch_flybase_gene_data(fbgn_ids)
        enriched_go = compute_go_enrichment(gene_data, background_go)
        for item in enriched_go[:10]:  # top 10
            go_info = go_info_given_id[item['GO_ID']]
            print(f"{item['GO_ID']}: input={item['Input_Count']} background={item['Background_Count']:4,d} "
                  f"p={item['p_value']:.3e}, q={item['q_value']:.3e}"
                  f"   {go_info['go_ns']}, {go_info['go_name']}")
        write_enriched_go_csv(enriched_go, out_fpath)
        break

results/Prosalpha3/CC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3_GO/CC_Full_Sample_Control_vs_Perturbation.csv
['CG33346', 'lectin-37Db', 'GstE10', 'CG45060', 'Jon65Aiii', 'Eip55E', 'Acbp5', 'CG9568', 'CG2064', 'tobi', 'CG30154', 'CG44013', 'Mal-A8', 'Cyp4p2']
['FBgn0053346', 'FBgn0053533', 'FBgn0063499', 'FBgn0266430', 'FBgn0035665', 'FBgn0000566', 'FBgn0035926', 'FBgn0032087', 'FBgn0033205', 'FBgn0261575', 'FBgn0050154', 'FBgn0264775', 'FBgn0033297', 'FBgn0033395']
GO:0019346: input=1 background=   1 p=1.046e-03, q=1.603e-02   biological_process, transsulfuration
GO:0004123: input=1 background=   1 p=1.046e-03, q=1.603e-02   molecular_function, cystathionine gamma-lyase activity
GO:0016846: input=1 background=   1 p=1.046e-03, q=1.603e-02   molecular_function, carbon-sulfur lyase activity
GO:0019343: input=1 background=   2 p=2.090e-03, q=1.923e-02   biological_process, cysteine biosynthetic process via cystathionine
GO:0000096: input=1 background=   2 p=2.090e-03, q=

In [22]:
fbgn_given_symbol['Bbd']

'FBgn0034512'

In [23]:
for symbol, fbgn in fbgn_given_symbol.items():
    if symbol not in obs_fbgn_given_symbol:
        for obs_symbol, obs_fbgn in obs_fbgn_given_symbol.items():
            if obs_fbgn == fbgn:
                print(fbgn, symbol, obs_symbol)

FBgn0039859 Mnat9 CG11539
FBgn0039887 Pos CG2053
FBgn0039890 Abcd1 ABCD
FBgn0052850 Rnf11 CG32850
FBgn0026262 Taf3 bip2
FBgn0026869 Tdg Thd1
FBgn0010391 SrpRalpha Gtp-bp
FBgn0003277 Polr2A RpII215
FBgn0030317 pkm CG1561
FBgn0030420 pira CG12717
FBgn0263005 Chpf CG43313
FBgn0030514 Mrgn1 CG9941
FBgn0030522 amrt CG11103
FBgn0030521 CtsB CtsB1
FBgn0026713 Prp16 l(1)G0007
FBgn0030612 Dbct CG5599
FBgn0030610 Cox17 CG9065
FBgn0030631 Prp5 CG6227
FBgn0030662 Chsy CG9220
FBgn0030687 Polr3A RpIIIC160
FBgn0030693 sordd1 CG8974
FBgn0052581 sordd2 CG32581
FBgn0030734 ERp44 CG9911
FBgn0030768 Nemp CG9723
FBgn0259923 Septin4 Sep4
FBgn0030793 Rai1 CG9125
FBgn0030801 Polr3I Rcp
FBgn0000617 Taf9 e(y)1
FBgn0030892 Swt1 CG7206
FBgn0052549 Nt5b CG32549
FBgn0030913 kairos CG6123
FBgn0030944 Hou CG6617
FBgn0085430 Dora CG34401
FBgn0031041 Pstk CG12788
FBgn0031060 Tcs4 CG14231
FBgn0031069 Abcd3 Pmp70
FBgn0031089 aspr CG9572
FBgn0031092 Ech1 CG9577
FBgn0052506 tbc CG32506
FBgn0052521 Mnr CG32521
FBgn0031170 A

In [24]:
for dpath in DE_dirs:
    out_dir = dpath + '_GO'
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    fpaths = glob.glob(f'{dpath}/*csv')
    for fpath in fpaths:
        print(fpath)
        out_fpath = os.path.join(out_dir, os.path.basename(fpath))
        symbols = [entry['symbol'] for entry in read_this_csv(fpath)]
        if not symbols:
            continue
        try:
            fbgn_ids = [fbgn_given_symbol[symbol] if symbol in fbgn_given_symbol else obs_fbgn_given_symbol[symbol] for symbol in symbols]
        except:
            print([symbol for symbol in symbols if symbol not in fbgn_given_symbol])
            fbgn_ids = [fbgn_given_symbol[symbol] for symbol in symbols if symbol in fbgn_given_symbol]
        gene_data = fetch_flybase_gene_data(fbgn_ids)
        enriched_go = compute_go_enrichment(gene_data, background_go)
        write_enriched_go_csv(enriched_go, out_fpath)

results/Prosalpha3/CC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/EB_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/ISC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/LFC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/VM_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/aEC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/dEC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/dMT_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/mEC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/pEC_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/ISC-EB_Full_Sample_Control_vs_Perturbation.csv
results/Prosalpha3/EB_Pert_wt_vs_Pert_het.csv
results/Prosalpha3/EB_Pert_wt_vs_Pert_hom.csv
results/Prosalpha3/EB_Pert_het_vs_Pert_hom.csv
results/Prosalpha3/EB_Pert_wt_vs_Pert_mut.csv
results/Prosalpha3/ISC_Pert_wt_vs_Pert_het.csv
results/Prosalpha3/ISC_Pert_wt_vs_Pert_hom.csv
results/Prosalpha3/ISC_Pert_

# Gene Summaries

In [25]:
import gzip

In [26]:
summary_fpath = '/g/stegle/hawkins/decode/annotation/dm6/flybase/best_gene_summary_current.tsv'
summary_given_fbgn = {}
for line in open(summary_fpath):
    if line.startswith('#'): #FBgn_ID'):
        summary_given_fbgn['header'] = line
    else:
        words = line.strip().split('\t')
        assert len(words) == 4, line
        fbgn, _, _, summary = words
        summary_given_fbgn[fbgn] = summary

In [27]:
summary_given_fbgn['header']

'#FBgn_ID\tGene_Symbol\tSummary_Source\tSummary\n'

In [28]:
summary_given_fbgn["FBgn0004647"]

'Essential signaling protein which has a major role in many developmental processes (PubMed:3935325). Functions as a receptor for membrane-bound ligands Delta and Serrate to regulate cell-fate determination (PubMed:10935637, PubMed:12909620, PubMed:15620650, PubMed:18243100). Upon ligand activation, and releasing from the cell membrane, the Notch intracellular domain (NICD) forms a transcriptional activator complex with Su(H) (Suppressor of hairless) and activates genes of the E(spl) complex (PubMed:7671825). Regulates oogenesis, the differentiation of the ectoderm and the development of the central and peripheral nervous system, eye, wing disk, muscles and segmental appendages such as antennae and legs, through lateral inhibition or induction (PubMed:11719214, PubMed:12369105, PubMed:3935325). Regulates neuroblast self-renewal, identity and proliferation through the regulation of bHLH-O proteins; in larval brains, involved in the maintenance of type II neuroblast self-renewal and iden

In [29]:
from collections import defaultdict

In [30]:
symbols_given_fbgn = {fbgn: [symbol] for symbol, fbgn in fbgn_given_symbol.items()}
for symbol, fbgn in obs_fbgn_given_symbol.items():
    if fbgn not in symbols_given_fbgn:
        symbols_given_fbgn[fbgn] = [symbol]
    elif symbol not in symbols_given_fbgn[fbgn]:
        symbols_given_fbgn[fbgn].append(symbol)

In [31]:
for dpath in DE_dirs:
    print('\n', dpath)
    out_dir = dpath + '_summaries'
    if not os.path.exists(out_dir):
        os.mkdir(out_dir)
    fpaths = glob.glob(f'{dpath}/*csv')
    for fpath in fpaths:
        print('.', end='')
        out_fpath = os.path.join(out_dir, os.path.basename(fpath))
        symbols = [entry['symbol'] for entry in read_this_csv(fpath)]
        if not symbols:
            continue
        fbgn_ids = [fbgn_given_symbol[symbol] if symbol in fbgn_given_symbol else obs_fbgn_given_symbol[symbol] for symbol in symbols]
        with open(out_fpath, 'w') as out:
            for fbgn in fbgn_ids:
                summary = 'No summary' if fbgn not in summary_given_fbgn else summary_given_fbgn[fbgn]
                out.write('\t'.join([fbgn] + symbols_given_fbgn[fbgn]) + '\n')
                out.write(summary + '\n')
                out.write('\n')


 results/Prosalpha3
.........................................................................................................................
 results/LamC
.......................................................................................